# Sweep タスクのポリシーを Colab で学習する

ローカルの `make train` と同じ `tools/sweep_train.py` を、同じ lock（`docker/envs/train/uv.lock`）から作った環境で実行します（docs/sweep.md の「Colab での学習」）。トークンは使いません。

**準備（ローカル）**
1. `make colab-data DATASET=sweep_floor` で `data/colab/sweep_floor.tar` を作り、ブラウザで Drive の `MyDrive/leisaac/` に上げる
2. 学習に使う commit を GitHub に push しておく

**実行**
1. ランタイムのタイプを GPU（L4 か A100）にする。T4 は使えない（flash-attn と bf16 に Ampere 以降が要る）
2. 下の「設定」を書き換え、上から順に実行する。Drive のマウントでは許可画面で承認する
3. 学習は Colab のローカルディスクに書き、チェックポイントができるたびに Drive の `MyDrive/leisaac/train/<TRAIN_NAME>/` に写す（Drive はシンボリックリンクを置けないので、lerobot の出力先にはしない）。接続が切れたら、同じ設定のまま上から実行し直すと、Drive の最後のチェックポイントから続ける
4. 最後のセルが `MyDrive/leisaac/<TRAIN_NAME>.tar` を作る。ブラウザでダウンロードし、ローカルで `make import-policy FILE=<tar> TRAIN_NAME=<TRAIN_NAME>` を実行する

**Drive の容量**: チェックポイント 1 つは SmolVLA で約 1.3GB（重み 0.87GB、optimizer などの学習の状態 0.4GB）。最後のもの以外は学習の状態を消すので、重みの分（約 0.87GB × 数）と tar の分が要る

In [ ]:
# 設定（ここだけ書き換える）
COMMIT = "origin/main"         # 学習に使う commit（push 済みのもの）。再現のため sha で指定するのがよい
DATASET = "sweep_floor"        # MyDrive/leisaac/<DATASET>.tar（make colab-data の出力）
TRAIN_POLICY = "act"           # act / smolvla / groot
TRAIN_STEPS = 50               # None ならポリシーごとの既定（tools/sweep_train.py）
TRAIN_BATCH = None             # None ならポリシーごとの既定
TRAIN_SEED = None              # None なら lerobot の既定（1000）
SAVE_FREQ = None               # チェックポイントの間隔（step）。None なら steps の 1/5
# lerobot-train にそのまま渡す引数（例: ["--policy.chunk_size=50"]）。SmolVLA の学習率を下げる期間は、sweep_train.py が steps に合わせる
EXTRA_ARGS = []
TRAIN_NAME = f"{DATASET}-{TRAIN_POLICY}"
KEEP_ALL_CHECKPOINTS = True    # False なら tar に最後のチェックポイントだけを入れる（学習曲線を描くなら True）

DRIVE_DIR = "/content/drive/MyDrive/leisaac"
WORK_DIR = "/content/leisaac"  # Colab のローカルディスク（セッションが終わると消える）
REPO_URL = "https://github.com/Chu-son/leisaac-docker"
UV_VERSION = "0.12.15"         # docker/Dockerfile の UV_IMAGE と合わせる

In [ ]:
# コマンドを実行して出力を表示する補助。失敗したら止める
import json, os, subprocess, sys

def sh(cmd, env=None, cwd=None):
    print(f"+ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", cwd=cwd, env={**os.environ, **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"失敗しました（終了コード {p.returncode}）: {cmd}")

def out(cmd):
    return subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True).stdout.strip()

## Drive と GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DATA_TAR = f"{DRIVE_DIR}/{DATASET}.tar"
RESULT_TAR = f"{DRIVE_DIR}/{TRAIN_NAME}.tar"
assert os.path.isfile(DATA_TAR), f"{DATA_TAR} がありません（make colab-data の出力を Drive に上げる）"
assert not os.path.exists(RESULT_TAR), f"{RESULT_TAR} は既にあります（TRAIN_NAME を変える）"
# 学習はローカルディスク (OUTPUT_DIR) に書き、チェックポイントを Drive (DRIVE_OUT) に写す。
# Drive に学習の状態 (training_state) まで写したチェックポイントがあれば、そこから続ける
OUTPUT_DIR = f"{WORK_DIR}/train/{TRAIN_NAME}"
DRIVE_OUT = f"{DRIVE_DIR}/train/{TRAIN_NAME}"
def drive_steps():
    d = f"{DRIVE_OUT}/checkpoints"
    return sorted(x for x in os.listdir(d) if x.isdigit()) if os.path.isdir(d) else []
RESUME_STEP = next((x for x in reversed(drive_steps()) if os.path.isdir(f"{DRIVE_OUT}/checkpoints/{x}/training_state")), None)
RESUME = RESUME_STEP is not None
print(f"{RESUME_STEP} step から続ける" if RESUME else "最初から学習する", DRIVE_OUT)

sh("nvidia-smi")
name, driver, cc = out("nvidia-smi --query-gpu=name,driver_version,compute_cap --format=csv,noheader").split(", ")
GPU = {"name": name, "driver": driver, "compute_cap": cc}
print(GPU)
assert float(cc) >= 8.0, "flash-attn と bf16 には Ampere 以降（L4、A100）が要る。ランタイムのタイプを変える"

## 環境（GitHub から clone し、lock から作る）

In [ ]:
REPO = f"{WORK_DIR}/repo"
VENV = f"{WORK_DIR}/venv"
PY = f"{VENV}/bin/python"
if not os.path.isdir(REPO):
    sh(f"git clone -q {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch -q origin && git -C {REPO} checkout -q --detach {COMMIT}")
REV = out(f"git -C {REPO} rev-parse HEAD")
print("commit", REV)

# lock と pyproject が食い違うと --locked が失敗する。Python 3.11 は uv が取得する
sh(f"{sys.executable} -m pip install -q uv=={UV_VERSION}")
sh(f"{sys.executable} -m uv sync --locked", cwd=f"{REPO}/docker/envs/train",
   env={"UV_PROJECT_ENVIRONMENT": VENV, "VIRTUAL_ENV": VENV, "UV_PYTHON_INSTALL_DIR": f"{WORK_DIR}/python"})

In [ ]:
# torch の CUDA と flash-attn が、この GPU とドライバで動くか（ローカルの make check の 6 番目と同じ確かめ方）
CHECK = """
from importlib.metadata import version
import torch
from flash_attn import flash_attn_func
print("torch", torch.__version__, "CUDA", torch.version.cuda, "lerobot", version("lerobot"), "flash-attn", version("flash-attn"))
assert torch.cuda.is_available(), "torch から GPU が見えない (ドライバが cu128 の torch に対応していない可能性)"
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
x = torch.randn(256, 256, device="cuda")
assert torch.isfinite(x @ x).all()
q = torch.randn(1, 128, 8, 64, device="cuda", dtype=torch.bfloat16)
assert flash_attn_func(q, q, q).shape == q.shape
torch.cuda.synchronize()
print("ENV OK")
"""
with open(f"{WORK_DIR}/env_check.py", "w") as f:
    f.write(CHECK)
sh(f"{PY} {WORK_DIR}/env_check.py")

## 学習

In [ ]:
DATASET_ROOT = f"{WORK_DIR}/data/lerobot/{DATASET}"
if not os.path.isdir(DATASET_ROOT):
    os.makedirs(f"{WORK_DIR}/data/lerobot", exist_ok=True)
    sh(f"tar -C {WORK_DIR}/data/lerobot -xf {DATA_TAR}")
assert os.path.isfile(f"{DATASET_ROOT}/meta/info.json"), f"{DATA_TAR} の中身が {DATASET}/ ではない"

import shlex, shutil, threading
args = f"--policy={TRAIN_POLICY} --dataset_root={DATASET_ROOT} --output_dir={OUTPUT_DIR}"
if RESUME:
    # Drive の最後のチェックポイントをローカルに戻し、そこから最初に決めた steps まで続ける
    # (データセットの場所は最初と同じ WORK_DIR の下なので、チェックポイントの train_config.json のままで読める)
    local = f"{OUTPUT_DIR}/checkpoints/{RESUME_STEP}"
    if not os.path.isdir(local):
        shutil.copytree(f"{DRIVE_OUT}/checkpoints/{RESUME_STEP}", local)
    if os.path.lexists(f"{OUTPUT_DIR}/checkpoints/last"):
        os.remove(f"{OUTPUT_DIR}/checkpoints/last")
    os.symlink(RESUME_STEP, f"{OUTPUT_DIR}/checkpoints/last")
    args += " --resume"
else:
    assert not os.path.exists(DRIVE_OUT), f"{DRIVE_OUT} は既にある (学習の状態がないので続けられない。TRAIN_NAME を変える)"
    if TRAIN_STEPS:
        args += f" --steps={TRAIN_STEPS}"
    if TRAIN_BATCH:
        args += f" --batch_size={TRAIN_BATCH}"
    if TRAIN_SEED is not None:
        args += f" --seed={TRAIN_SEED}"
    if SAVE_FREQ:
        args += f" --save_freq={SAVE_FREQ}"
    if EXTRA_ARGS:
        args += " -- " + " ".join(shlex.quote(a) for a in EXTRA_ARGS)

def sync():
    """書き終わったチェックポイント (last が指すもの以前) を Drive に写す。学習の状態は最新の 1 つだけに残す"""
    d = f"{OUTPUT_DIR}/checkpoints"
    if not os.path.islink(f"{d}/last"):
        return
    last = os.readlink(f"{d}/last")
    os.makedirs(f"{DRIVE_OUT}/checkpoints", exist_ok=True)
    for x in sorted(x for x in os.listdir(d) if x.isdigit() and x <= last):
        dst = f"{DRIVE_OUT}/checkpoints/{x}"
        if not os.path.isdir(dst):
            shutil.copytree(f"{d}/{x}", f"{dst}.part")
            os.rename(f"{dst}.part", dst)
            print(f"[Drive] {x} を写した", flush=True)
    for x in drive_steps()[:-1]:
        shutil.rmtree(f"{DRIVE_OUT}/checkpoints/{x}/training_state", ignore_errors=True)

def sync_loop(stop):
    while not stop.wait(120):
        try:
            sync()
        except Exception as e:  # 写す途中の失敗は次の回にやり直す
            print("[Drive] 写せなかった:", e, flush=True)
stop = threading.Event()
threading.Thread(target=sync_loop, args=(stop,), daemon=True).start()
# モデルの重み（Hub と torchvision）は Colab のローカルディスクに置く
try:
    sh(f"{PY} {REPO}/tools/sweep_train.py {args}",
       env={"PYTHONPATH": f"{REPO}/tools", "HF_HOME": f"{WORK_DIR}/cache", "TORCH_HOME": f"{WORK_DIR}/cache/torch",
            "VIRTUAL_ENV": VENV, "PATH": f"{VENV}/bin:{os.environ['PATH']}"})
finally:
    stop.set()
sync()


## Drive に保存

In [ ]:
# Drive の DRIVE_OUT (すべてのチェックポイント) に、sweep_train.json、colab.json、checkpoints/last を足して tar にする
# (make import-policy は ./sweep_train.json があることを確かめる)。colab.json に、使った commit と GPU を残す
shutil.copy(f"{OUTPUT_DIR}/sweep_train.json", f"{DRIVE_OUT}/sweep_train.json")
with open(f"{DRIVE_OUT}/colab.json", "w") as f:
    json.dump({"commit": REV, "gpu": GPU, "train_policy": TRAIN_POLICY, "train_steps": TRAIN_STEPS,
               "train_batch": TRAIN_BATCH, "train_seed": TRAIN_SEED, "save_freq": SAVE_FREQ, "extra_args": EXTRA_ARGS,
               "resumed_from": RESUME_STEP, "dataset": DATASET}, f, indent=2)
last = drive_steps()[-1]
members = "./checkpoints" if KEEP_ALL_CHECKPOINTS else f"./checkpoints/{last}"
# Drive にはシンボリックリンクを置けないので、checkpoints/last はローカルで作って足す
link_dir = f"{WORK_DIR}/link/{TRAIN_NAME}"
os.makedirs(f"{link_dir}/checkpoints", exist_ok=True)
if not os.path.lexists(f"{link_dir}/checkpoints/last"):
    os.symlink(last, f"{link_dir}/checkpoints/last")
# 書き込みの途中で切れたものを残さないよう、別名で書いてから名前を変える
sh(f"tar -C {DRIVE_OUT} -cf {RESULT_TAR}.part ./sweep_train.json ./colab.json {members} && "
   f"tar -C {link_dir} -rf {RESULT_TAR}.part ./checkpoints/last && mv {RESULT_TAR}.part {RESULT_TAR}")
sh(f"ls -lh {RESULT_TAR}")
print(f"ダウンロードしたら: make import-policy FILE=<ダウンロードした {TRAIN_NAME}.tar> TRAIN_NAME={TRAIN_NAME}")
print(f"tar を取り込んだ後は、Drive の {DRIVE_OUT} を消してよい")
